<a href="https://colab.research.google.com/github/hyeonukim/IndoorLocalization/blob/main/indoor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
print('hello')

hello


In [1]:
import subprocess, sys

gpu_info = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(gpu_info.stdout)

if 'A100' not in gpu_info.stdout:
    print('⚠️  WARNING: A100 not detected. Go to Runtime > Change runtime type > A100 GPU.')
else:
    print('✅ A100 GPU confirmed.')

Mon Mar 30 02:42:55 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   31C    P0             55W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [6]:
!pip install -q transformers accelerate huggingface_hub datasets
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu118
!pip install -q open3d matplotlib scipy scikit-learn tqdm einops
!pip install -q Pillow opencv-python-headless h5py faiss-gpu

# huggingface_hub CLI for dataset download
!pip install -q 'huggingface_hub[cli]'

ERROR: Could not find a version that satisfies the requirement faiss-gpu (from versions: none)
ERROR: No matching distribution found for faiss-gpu


In [7]:
import os, sys, json, time, random, shutil, re
from pathlib import Path
from collections import defaultdict
from tqdm import tqdm

import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
import matplotlib.pyplot as plt
import cv2

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {DEVICE}')
print(f'PyTorch: {torch.__version__}')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

Device : cuda
PyTorch: 2.10.0+cu128


In [8]:
ROOT        = Path('/content/vggt_nyc')
DATA_DIR    = ROOT / 'data' / 'nyc_indoor_vpr'
RESULTS_DIR = ROOT / 'results'
MODELS_DIR  = ROOT / 'models'
VIS_DIR     = ROOT / 'visualizations'

for d in [DATA_DIR, RESULTS_DIR, MODELS_DIR, VIS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('✅ Project directories ready:')
for d in [ROOT, DATA_DIR, RESULTS_DIR, MODELS_DIR, VIS_DIR]:
    print(f'  {d}')

✅ Project directories ready:
  /content/vggt_nyc
  /content/vggt_nyc/data/nyc_indoor_vpr
  /content/vggt_nyc/results
  /content/vggt_nyc/models
  /content/vggt_nyc/visualizations


In [9]:
# ── Option A: Download from HuggingFace (recommended) ────────────
# You may need to log in: huggingface-cli login
# The dataset is at: https://huggingface.co/datasets/ai4ce/NYC-Indoor-VPR-Data

from huggingface_hub import snapshot_download

try:
    print('Downloading NYC-Indoor-VPR from HuggingFace ...')
    print('This is ~36,000 images — may take 10-20 min on first run.')
    snapshot_download(
        repo_id='ai4ce/NYC-Indoor-VPR-Data',
        repo_type='dataset',
        local_dir=str(DATA_DIR),
        ignore_patterns=['*.git*'],
    )
    print('✅ Download complete.')
except Exception as e:
    print(f'HuggingFace download failed: {e}')
    print('Alternative: mount Google Drive if dataset is already there.')
    print('  from google.colab import drive')
    print('  drive.mount("/content/drive")')
    print('  !cp -r /content/drive/MyDrive/nyc_indoor_vpr/* /content/vggt_nyc/data/nyc_indoor_vpr/')

This is ~36,000 images — may take 10-20 min on first run.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

✅ Download complete.


In [11]:
# Explore the downloaded folder structure
all_images = sorted(DATA_DIR.rglob('*.jpg')) + sorted(DATA_DIR.rglob('*.png'))
print(f'Total images found: {len(all_images)}')
print('\nSample filenames (first 10):')
for p in all_images[:10]:
    print(f'  {p.relative_to(DATA_DIR)}')

# Show top-level folder structure
print('\nTop-level folders:')
for d in sorted(DATA_DIR.iterdir()):
    if d.is_dir():
        n = len(list(d.rglob('*.jpg')))
        print(f'  {d.name}/  ({n} images)')

Total images found: 0

Sample filenames (first 10):

Top-level folders:
  .cache/  (0 images)


In [12]:
def parse_nyc_filename(path: Path) -> dict | None:
    """
    Parse NYC-Indoor-VPR filename to extract metadata.

    Expected format (verify against your downloaded filenames):
        <scene>_<traversal>_<x>_<y>_<frame>.jpg

    Returns dict with keys: scene, traversal, x, y, frame, path
    Returns None if filename does not match the expected pattern.

    ── HOW TO ADAPT ──────────────────────────────────────────────
    After downloading, run:
        sample = list(DATA_DIR.rglob('*.jpg'))[:5]
        for p in sample: print(p.name)
    Adjust the regex pattern below to match the actual format.
    ──────────────────────────────────────────────────────────────
    """
    name = path.stem  # filename without extension

    # Pattern 1: scene_traversal_x_y_frame  (e.g., oculus_01_123_456_0042)
    m = re.match(r'^(.+?)_(\d+)_(-?\d+\.?\d*)_(-?\d+\.?\d*)_(\d+)$', name)
    if m:
        return {
            'scene'    : m.group(1),
            'traversal': int(m.group(2)),
            'x'        : float(m.group(3)),
            'y'        : float(m.group(4)),
            'frame'    : int(m.group(5)),
            'path'     : path,
        }

    # Pattern 2: location encoded differently — inspect and add here
    # m = re.match(r'...', name)

    # Fallback: try to extract from parent folder names
    parts = path.parts
    # e.g., data/scene_name/traversal_id/image.jpg
    if len(parts) >= 3:
        return {
            'scene'    : parts[-3] if len(parts) >= 3 else 'unknown',
            'traversal': parts[-2] if len(parts) >= 2 else 'unknown',
            'x'        : None,
            'y'        : None,
            'frame'    : None,
            'path'     : path,
        }
    return None


# Parse all images and build metadata table
metadata = []
parse_failures = 0
for p in all_images:
    m = parse_nyc_filename(p)
    if m:
        metadata.append(m)
    else:
        parse_failures += 1

print(f'Parsed {len(metadata)} images successfully.')
print(f'Parse failures: {parse_failures} (check parse_nyc_filename regex)')

# Summary by scene
scenes = defaultdict(list)
for m in metadata:
    scenes[m['scene']].append(m)

print(f'\nScenes found: {len(scenes)}')
for scene, imgs in sorted(scenes.items()):
    traversals = set(str(m['traversal']) for m in imgs)
    print(f'  {scene}: {len(imgs)} images, {len(traversals)} traversals ({sorted(traversals)})')

Parsed 0 images successfully.
Parse failures: 0 (check parse_nyc_filename regex)

Scenes found: 0


In [13]:
def build_db_query_split(metadata: list, query_traversals_per_scene: int = 2) -> tuple:
    """
    Split NYC-Indoor-VPR into database and query sets.

    Strategy:
      - For each scene, sort traversals chronologically.
      - Earliest traversal(s) → database (reference map)
      - Later traversal(s)    → queries  (test set)

    This is the natural VPR setup: localize later visits
    against an earlier reference map.

    Args:
        metadata                : list of parsed image dicts
        query_traversals_per_scene: how many later traversals to use as queries

    Returns:
        db_images, query_images : lists of metadata dicts
    """
    db_images    = []
    query_images = []

    scenes = defaultdict(list)
    for m in metadata:
        scenes[m['scene']].append(m)

    for scene, imgs in scenes.items():
        # Group by traversal
        traversal_groups = defaultdict(list)
        for m in imgs:
            traversal_groups[str(m['traversal'])].append(m)

        sorted_traversals = sorted(traversal_groups.keys())
        if len(sorted_traversals) < 2:
            # Not enough traversals to form query set — all go to DB
            db_images.extend(imgs)
            continue

        n_query = min(query_traversals_per_scene, len(sorted_traversals) - 1)
        db_traversals    = sorted_traversals[:-n_query]
        query_traversals = sorted_traversals[-n_query:]

        for t in db_traversals:
            db_images.extend(traversal_groups[t])
        for t in query_traversals:
            query_images.extend(traversal_groups[t])

    print(f'Database images : {len(db_images)}')
    print(f'Query images    : {len(query_images)}')
    return db_images, query_images


db_metadata, query_metadata = build_db_query_split(metadata, query_traversals_per_scene=2)

# Extract path lists for convenience
db_paths    = [m['path'] for m in db_metadata]
query_paths = [m['path'] for m in query_metadata]

Database images : 0
Query images    : 0


In [14]:
def show_image_grid(image_paths: list, title: str = '', n: int = 8, cols: int = 4):
    n = min(n, len(image_paths))
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3))
    axes = np.array(axes).flatten()
    for i, ax in enumerate(axes):
        if i < n:
            img = Image.open(image_paths[i]).convert('RGB')
            ax.imshow(img)
            ax.set_title(Path(image_paths[i]).name[:20], fontsize=6)
        ax.axis('off')
    fig.suptitle(title, fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(VIS_DIR / f'{title.replace(" ", "_")}.png', dpi=100, bbox_inches='tight')
    plt.show()


def plot_scene_trajectory(metadata: list, scene: str,
                           save_path: Path = None):
    """
    Plot camera trajectories for all traversals in a scene
    using the topometric (x, y) coordinates.
    """
    scene_imgs = [m for m in metadata if m['scene'] == scene and m['x'] is not None]
    if not scene_imgs:
        print(f'No parseable topometric coords for scene: {scene}')
        return

    traversal_groups = defaultdict(list)
    for m in scene_imgs:
        traversal_groups[str(m['traversal'])].append(m)

    fig, ax = plt.subplots(figsize=(8, 6))
    cmap = plt.cm.tab10

    for i, (trav, imgs) in enumerate(sorted(traversal_groups.items())):
        xs = [m['x'] for m in sorted(imgs, key=lambda m: m.get('frame', 0))]
        ys = [m['y'] for m in sorted(imgs, key=lambda m: m.get('frame', 0))]
        ax.plot(xs, ys, color=cmap(i), linewidth=1.5, label=f'Traversal {trav}', alpha=0.8)
        ax.scatter(xs[0], ys[0], color=cmap(i), marker='o', s=60, zorder=5)

    ax.set_title(f'Trajectory Map — {scene}', fontweight='bold', fontsize=12)
    ax.set_xlabel('X (topometric)')
    ax.set_ylabel('Y (topometric)')
    ax.legend(fontsize=8, loc='best')
    ax.grid(alpha=0.3)
    plt.tight_layout()
    save_path = save_path or VIS_DIR / f'trajectory_{scene}.png'
    plt.savefig(save_path, dpi=120, bbox_inches='tight')
    plt.show()


# Sample images per building type
if db_paths:
    show_image_grid([str(p) for p in random.sample(db_paths, min(8, len(db_paths)))],
                    'NYC Indoor VPR - Database Samples')

# Plot trajectory for one scene (e.g., Oculus or whichever scene loads first)
if scenes:
    first_scene = list(scenes.keys())[0]
    plot_scene_trajectory(metadata, first_scene)

# 2


In [15]:
import faiss
from transformers import AutoImageProcessor, AutoModel

class DINOv2Retrieval:
    """
    Dense retrieval using DINOv2 (facebook/dinov2-base).
    Uses FAISS IndexFlatIP for fast cosine similarity search over 36K images.
    """
    def __init__(self, model_name: str = 'facebook/dinov2-base', device: str = DEVICE):
        print(f'Loading DINOv2 ({model_name}) ...')
        self.processor = AutoImageProcessor.from_pretrained(model_name)
        self.model     = AutoModel.from_pretrained(model_name).to(device)
        self.model.eval()
        self.device  = device
        self.index   = None
        self.db_meta = None
        print('✅ DINOv2 loaded.')

    @torch.no_grad()
    def embed_images(self, image_paths: list, batch_size: int = 64) -> np.ndarray:
        """Embed a list of image paths. Returns (N, D) float32 numpy array."""
        all_embeddings = []
        for i in tqdm(range(0, len(image_paths), batch_size), desc='Embedding'):
            batch = image_paths[i: i + batch_size]
            images = []
            for p in batch:
                try:
                    images.append(Image.open(p).convert('RGB'))
                except Exception:
                    images.append(Image.new('RGB', (224, 224)))
            inputs = self.processor(images=images, return_tensors='pt').to(self.device)
            cls_embs = self.model(**inputs).last_hidden_state[:, 0, :]
            cls_embs = F.normalize(cls_embs, dim=-1)
            all_embeddings.append(cls_embs.cpu().float().numpy())
        return np.vstack(all_embeddings)

    def build_faiss_index(self, db_metadata: list):
        """
        Build a FAISS flat index over all database images.
        Caches embeddings to disk — only computed once.
        """
        cache_emb   = MODELS_DIR / 'dinov2_db_embs.npy'
        cache_paths = MODELS_DIR / 'dinov2_db_paths.json'

        if cache_emb.exists() and cache_paths.exists():
            print('Loading cached DINOv2 database embeddings ...')
            embs = np.load(cache_emb)
            with open(cache_paths) as f:
                saved_paths = json.load(f)
            # Verify cache matches current db_metadata
            current_paths = [str(m['path']) for m in db_metadata]
            if saved_paths == current_paths:
                self.db_meta = db_metadata
            else:
                print('Cache mismatch — recomputing.')
                embs = self._compute_and_cache(db_metadata, cache_emb, cache_paths)
        else:
            embs = self._compute_and_cache(db_metadata, cache_emb, cache_paths)

        # Build FAISS inner product index (cosine similarity, since embs are normalized)
        d = embs.shape[1]
        self.index   = faiss.IndexFlatIP(d)
        self.index.add(embs.astype(np.float32))
        self.db_meta = db_metadata
        print(f'FAISS index built: {self.index.ntotal} vectors, dim={d}')

    def _compute_and_cache(self, db_metadata, cache_emb, cache_paths):
        print(f'Computing DINOv2 embeddings for {len(db_metadata)} images ...')
        paths = [str(m['path']) for m in db_metadata]
        embs  = self.embed_images(paths)
        np.save(cache_emb, embs)
        with open(cache_paths, 'w') as f:
            json.dump(paths, f)
        print(f'Embeddings cached: {cache_emb}')
        self.db_meta = db_metadata
        return embs

    def retrieve(self, query_path: str, top_k: int = 25) -> list:
        """
        Retrieve top-K database images for a single query.
        Returns list of (db_metadata_dict, similarity_score) tuples.
        """
        q_emb = self.embed_images([query_path])  # (1, D)
        scores, indices = self.index.search(q_emb.astype(np.float32), top_k)
        return [(self.db_meta[i], float(scores[0][j]))
                for j, i in enumerate(indices[0])]

    def retrieve_batch(self, query_paths: list, top_k: int = 25,
                        batch_size: int = 64) -> list:
        """
        Batch retrieval for all queries at once (much faster than one-by-one).
        Returns list of lists: [ [(db_meta, score), ...], ... ] per query.
        """
        q_embs = self.embed_images(query_paths, batch_size=batch_size)
        scores, indices = self.index.search(q_embs.astype(np.float32), top_k)
        results = []
        for i in range(len(query_paths)):
            row = [(self.db_meta[idx], float(scores[i][j]))
                   for j, idx in enumerate(indices[i])]
            results.append(row)
        return results


retrieval_model = DINOv2Retrieval()

# Build index (cached after first run)
if db_metadata:
    retrieval_model.build_faiss_index(db_metadata)

ModuleNotFoundError: No module named 'faiss'